# Optimizing modeling code with Popcorn

[`modeling_butter.py`](./modeling_butter.py) is a hybrid Wall-Attention / attention LM in plain PyTorch. Ops live in an `OPS` namespace; swap in `popcorn.kernels` without changing the model.

In [ ]:
from types import SimpleNamespace

import torch

from modeling_butter import OPS, ButterConfig, ButterForCausalLM
from popcorn import kernels
from popcorn.bench import compare, report

torch.manual_seed(0)
device, dtype = "cuda", torch.float32  # wall_attn gate grads need fp32
config = ButterConfig()
batch = {
    "input_ids": torch.randint(config.vocab_size, (2, 512), device=device),
    "labels": torch.randint(config.vocab_size, (2, 512), device=device),
}

baseline = ButterForCausalLM(config, ops=OPS).to(device, dtype)
POPCORN_OPS = SimpleNamespace(
    rms_norm=kernels.rms_norm,
    rope=kernels.rope,
    swiglu=kernels.swiglu,
    wall_attn=kernels.wall_attn["popcorn"],  # first-party wall kernel
    linear_cross_entropy=kernels.linear_cross_entropy,
)
model = ButterForCausalLM(config, ops=POPCORN_OPS).to(device, dtype)
model.load_state_dict(baseline.state_dict())


def forward(m):
    def run(**batch):
        with torch.no_grad():
            return m(**batch)

    return run


def train_step(m):
    def run(**batch):
        loss = m(**batch)
        loss.backward()
        m.zero_grad(set_to_none=True)
        return loss.detach()

    return run


kw = dict(mine="popcorn", reference="baseline", assert_rel=0.02)
report(compare(forward(model), forward(baseline), batch, backward=False, repeats=3), "forward", **kw)
report(compare(train_step(model), train_step(baseline), batch, backward=False, repeats=2), "train step", **kw)

## Forcing a backend

In [ ]:
x = torch.randn(2, 512, config.hidden_size, device=device, dtype=dtype)
weight = torch.ones(config.hidden_size, device=device, dtype=dtype)

kernels.rms_norm(x, weight)
kernels.rms_norm(x, weight, backend="torch")
kernels.rms_norm["liger"](x, weight)
with kernels.rms_norm["torch"]:
    kernels.rms_norm(x, weight)

kernels.rms_norm